# 3.6 Kernel 整体框架：核间切块与负载均衡

Tiling 产出的「施工图纸」（TilingData + TilingKey）随任务一起下发到 Device。Kernel 这边开始动工：先按 SPMD 方式把任务铺到多个组合核上（核间切块），再在每个核内部沿 S2 轴推进循环（核内切块，下一节）。

![Kernel 层次](images/fia_kernel_hierarchy.svg)

## 1. Kernel 的入参

一个 Kernel 接口的入参包括：**输入、输出、workspace、tilingData**。

```cpp
extern "C" __global__ __aicore__ void fused_infer_attention_score(
    GM_ADDR query, GM_ADDR key, GM_ADDR value, GM_ADDR attenMask,
    GM_ADDR attentionOut, GM_ADDR workspace, GM_ADDR tiling)
{
    // SPMD：每个核执行相同代码，用 block_idx 区分自己的任务
}
```

对照 3.2 节：接口层的十几个张量与属性，到这里浓缩成「张量地址 + workspace + tilingData」——**属性类信息已经全部被 Tiling 消化进 TilingData**。

## 2. SPMD：同一份代码，不同的数据

**SPMD（Single Program Multiple Data）**：多核并行时，每个核上运行的代码相同，但处理的数据不同。并行的条件是**多核间的数据可以独立运算**；若多核间数据有依赖，则最后还需要一次**核间同步**。

FA 的核间独立性由「按 B / N2 / G / S1 外层切块」保证——不同任务块计算的是不同（batch、KV 头、Q 头组、S1 段）的输出行，互不依赖。

## 3. Kernel 计算流：两级切块

```text
核间切块（多核并行）
   按 B、N2、S1 轴把任务切成 totalSize 份，
   splitFactorSize 份一组派给每个组合核
        ▼
核内切块（每核内的循环）
   对 K、V 的 S2 轴切块，逐块推进四拍循环
   （mm1 → vector1 → mm2 → vector2）
```

### 为什么 K、V 必须按完整 S 轴加载？

Q 与 Key/Value 有本质区别：**Key、Val 一定是按完整的 S 轴加载**——Softmax 沿 S2 方向归一化，一个 Q 块必须「见到」该头全部 KV 才能算对分母 l。因此核间只切 Q 的 S1，**KV 的 S2 只能留在核内循环**。

### offset 计算

每个核根据 `block_idx` 算出自己负责的任务区间，再换算成 GM 地址偏移；GQA 场景下要按 `n2oIdx` 找到当前 KV 组。**非 GQA 场景下 `runParam.n2oIdx` 与 `nIdx` 相等**（N2 = N1，每个 Q 头都有自己专属的 KV）。

![核间切分](images/fia_tiling_split.svg)

## 4. 负载均衡：把任务均摊到核

回到 Tiling ④ 的两个量：

```text
totalSize     = B × N2 × G × ceil(S1 / s1BasicBlock)   // 总任务块数
actualUsedAiv = min(totalSize, aivNum)                   // 实际动用的 AIV 数
splitFactorSize = ceil(totalSize / actualUsedAiv)         // 每核领到的块数
```

Kernel 侧对应逻辑：核 `i` 领取任务区间 `[i × splitFactorSize, (i+1) × splitFactorSize)`，逐块处理。

**两个工程细节**：

1. **任务不足时少用核**：`totalSize < aivNum` 时只启动 `totalSize` 个组合，避免空转；
2. **尾块不均**：`totalSize` 不能整除时，前面的核多领一块——这正是 Tiling 选择「向上取整切分」、确保最差基本块性能最优的原因。

> 第 7~9 章的性能优化实战会反复回到这里：**负载均衡是 FA 性能优化的第一大项**（均衡性分析是常用性能优化步骤的第一步）。

## 5. Kernel 代码框架

每个组合核上的流程骨架：

```cpp
if (block_idx >= blockDim) return;            // 越界保护

// ① 从 GM 取出 TilingData 并解析
GET_TILING_DATA_WITH_STRUCT(...)              // 展开成局部变量

// ② 算出本核负责的任务区间（核间切块 + 负载均衡）
auto taskRange = CalcTaskRange(block_idx, totalSize, splitFactorSize);

// ③ 逐任务块处理
for (task : taskRange) {
    // 解析该块的 B/N/S1 偏移（含 GQA 的 n2oIdx 映射）
    // 核内对 S2 轴切块 → 四拍循环（下一节）
}
```

![Kernel 循环](images/fia_kernel_loop.svg)

## 6. 本节小结

- Kernel 入参四件套：输入、输出、workspace、tilingData；属性已被 Tiling 消化。
- SPMD + `block_idx` 区分任务；FA 核间数据独立，无需核间同步。
- 核间按 **B、N2、S1** 切块；**KV 必须完整 S 轴加载**（Softmax 归一化的硬约束），S2 留给核内循环。
- 负载均衡 = `ceil(totalSize / actualUsedAiv)` 份一组；任务不足少用核，不整除前面的核多领。

## 小测验

1. Kernel 入参里的 `tiling` 是什么？它和接口层的 `numHeads`、`inputLayout` 等属性是什么关系？
2. 为什么核间切块不把 KV 的 S2 也切给不同核？（提示：Softmax 的归一化方向）
3. GQA 场景下 `n2oIdx` 与 `nIdx` 有什么区别？非 GQA 时它们为什么相等？
4. `totalSize = 100`、`aivNum = 50`，splitFactorSize 是多少？若 `totalSize = 99` 呢，哪个核多干活？